In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from modulos.preparacao import preparar_dados

Processando moeda: AAVEBTC
Processando moeda: AAVEUSDT
Processando moeda: ACMUSDD
Processando moeda: ADAUSDT
Processando moeda: BNBUSDT
Processando moeda: BNTUSDT
Processando moeda: CVTBTC
Processando moeda: DOGEBTC
Processando moeda: ETCETH
Processando moeda: USDPUSDT

Moeda: AAVEBTC
  Fold 1: Train shape = (2482, 23),  Val shape = (2482, 23)
  Fold 2: Train shape = (4964, 23),  Val shape = (2482, 23)
  Fold 3: Train shape = (7446, 23),  Val shape = (2482, 23)
  Fold 4: Train shape = (9928, 23),  Val shape = (2482, 23)
  Fold 5: Train shape = (12410, 23),  Val shape = (2482, 23)

Moeda: AAVEUSDT
  Fold 1: Train shape = (4208, 23),  Val shape = (4204, 23)
  Fold 2: Train shape = (8412, 23),  Val shape = (4204, 23)
  Fold 3: Train shape = (12616, 23),  Val shape = (4204, 23)
  Fold 4: Train shape = (16820, 23),  Val shape = (4204, 23)
  Fold 5: Train shape = (21024, 23),  Val shape = (4204, 23)

Moeda: ACMUSDD
  Fold 1: Train shape = (351, 23),  Val shape = (350, 23)
  Fold 2: Train sha

In [ ]:

"""
treinamento.py

Etapa do trabalho: 9

Este script realiza o treinamento e a avaliação dos modelos para cada moeda:

1. Carrega os folds preparados no arquivo de preparação.
2. Define as predictors de entrada (predictors) e a variável alvo (outcome).
3. Treina um modelo MLP para cada fold, avaliando seu desempenho com métricas como MSE, MAE e R².
4. Treina também modelos de regressão linear e polinomial (graus 2 a 10) para comparação.
5. Gera gráficos de dispersão, curvas de predição e imprime erros estatísticos dos modelos.
6. Simula uma estratégia de investimento baseada nas previsões, partindo de U$ 1.000,
   e calcula o lucro acumulado em comparação com os modelos de regressão.
"""

# Prepara os folds temporais para cada moeda
folds_por_moeda = preparar_dados(n_splits=5)

# Define os predictors de entrada e a variável alvo
predictors = [
    'media_movel_7d', 'std_7d', 'momentum_7d', 'retorno_pct_7d',
    'volume_2_7d', 'taker_ratio', 'buy_pressure', 'volume_volatilidade_ratio',
    'dia_da_semana'
]
outcome = 'close'

# Dicionário para armazenar os resultados de todas as moedas
resumo_total = {}

# Loop principal para percorrer cada moeda
for coin, folds in folds_por_moeda.items():
    print(f"\n=== Moeda: {coin} ===")
    registros = []

    # Para cada fold de validação
    for fold_idx, split in enumerate(folds, start=1):
        # Extrai treino e validação a partir dos índices do fold
        X_train = split['train'][predictors].values
        y_train = split['train'][outcome].values
        X_val   = split['val'][predictors].values
        y_val   = split['val'][outcome].values

        # Define os modelos a serem comparados
        modelos = {
            'MLP': MLPRegressor(hidden_layer_sizes=(50,100,50), max_iter=800, random_state=1),
            'Linear': LinearRegression()
        }
        # Adiciona regressões polinomiais de grau 2 a 10
        for grau in range(2, 11):
            modelos[f'Poly_{grau}'] = make_pipeline(
                PolynomialFeatures(degree=grau, include_bias=False),
                LinearRegression()
            )

        # Treina e avalia cada modelo neste fold
        for nome, modelo in modelos.items():
            modelo.fit(X_train, y_train)        # Ajusta o modelo aos dados de treino
            y_pred = modelo.predict(X_val)      # Prediz sobre os dados de validação

            # Calcula métricas de erro
            mse = mean_squared_error(y_val, y_pred)
            mae = mean_absolute_error(y_val, y_pred)
            r2  = r2_score(y_val, y_pred)

            # Simula investimento: US$1000 iniciais, reinvestindo quando previsão > fechamento
            sinais   = (y_pred[1:] > y_val[:-1]).astype(int)
            retornos = np.where(sinais, y_val[1:] / y_val[:-1], 1.0)
            profit   = 1000 * np.cumprod(retornos)[-1]

            # Armazena resultados deste fold e modelo
            registros.append({
                'fold': fold_idx,
                'modelo': nome,
                'mse': mse,
                'mae': mae,
                'r2': r2,
                'profit': profit
            })

    # Converte a lista em DataFrame para facilitar análise
    df_resumo = pd.DataFrame(registros)
    resumo_total[coin] = df_resumo

    # Exibe a média das métricas e do lucro para cada modelo, ordenado por maior lucro
    print(
        df_resumo
        .groupby('modelo')[['mse','mae','r2','profit']]
        .mean()
        .sort_values('profit', ascending=False)
    )


Processando moeda: AAVEBTC
Processando moeda: AAVEUSDT
Processando moeda: ACMUSDD
Processando moeda: ADAUSDT
Processando moeda: BNBUSDT
Processando moeda: BNTUSDT
Processando moeda: CVTBTC
Processando moeda: DOGEBTC
Processando moeda: ETCETH
Processando moeda: USDPUSDT

=== Moeda: AAVEBTC ===
